# Is eICU's value-dominance a pooling artifact?

**The claim under test.** On the matched 12-analyte panel, database explains
more of the variation in the decomposition's composition than outcome does
(50.1% vs 33.3% for unique-to-value). eICU-CRD is far more value-dominated:
72.0% unique-to-value against MIMIC-IV's 45.1%, or equivalently 28.0% vs 54.9%
of information *not* explained by lab values alone.

**The confound.** MIMIC-IV is one hospital; eICU-CRD pools 198 (in this cohort).
If ordering practice is site-specific, pooling can wash out structure signal for
purely statistical reasons — different hospitals' draw-timing patterns cancel
when averaged — which would push eICU toward value-dominance without any
clinical difference in how the two systems behave. Earlier evidence is
suggestive: conditioning on hospital raised eICU's mortality structure signal by
~10% relative (`results/ijoint_cross_target_normalization.json`).

**Two things this design has to control.**

1. **Discretisation.** The codebook is fit **once on the full pooled cohort** and
   applied everywhere. Every hospital and every control sample uses identical
   level definitions, so only the composition of the joint table varies, never
   the channel definition.

2. **Sample size.** A hospital has ~1,000–3,800 patients against the pooled
   120,266, and plug-in PID atoms are biased differently at those two scales.
   Comparing a hospital to the pooled cohort would confound heterogeneity with
   sample size. So every hospital is compared against **size-matched random
   draws from the whole cohort** — same n, but crossing hospitals. The
   difference between "within one hospital" and "same-size random slice of many
   hospitals" is then the pooling effect, with n held fixed.

**Resolution.** k=13 is not estimable here: a median eligible hospital would
have 4.4 patients per joint cell. This runs at **k=5** (50 cells, ~30 per cell
at the median eligible hospital), so its numbers are *not* comparable to the
k=13 headline. The pooled cohort is therefore also recomputed at k=5, and that
— not the headline — is the reference. MIMIC-IV at k=5 is included for context.

In [1]:
import sys
sys.path.insert(0, "../src")

import json, time
import numpy as np
import pandas as pd

import mimic_channels as mimic
import eicu_channels as eicu
import quantize as qz
import pid_broja as pb

pd.set_option("display.width", 200)

K = 5                      # forced by within-hospital n, NOT the headline k=13
SEEDS = [0, 1, 2]          # codebook seeds
N_DRAWS = 5                # size-matched random draws per hospital
MIN_H = 1000               # minimum hospital size
ATOMS = ("red", "u_val", "u_str", "syn")
TARGETS = ["mortality", "gender_female", "age_ge65", "race_binary"]

MG = 50960
P12E = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
        "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}

cc_e = eicu.complete_case(labname_scope=P12E)
cc_m = mimic.complete_case(itemid_scope=set(mimic.CORE) - {MG})
_, ecoh = eicu._pull()
hosp = ecoh.set_index("patientunitstayid")["hospitalid"].reindex(cc_e.groups).to_numpy()
demo_e, _ = eicu.demographic_targets()
demo_m, _ = mimic.demographic_targets()

vc = pd.Series(hosp).value_counts()
BIG = vc[vc >= MIN_H].index.tolist()
print(f"eICU matched-12 cohort n={len(cc_e.y):,} across {vc.size} hospitals")
print(f"{len(BIG)} hospitals with n>={MIN_H}, covering {vc[BIG].sum():,} "
      f"({vc[BIG].sum()/len(hosp):.1%}) of the cohort")
print(f"k={K} -> {K*K*2} joint cells")

eICU matched-12 cohort n=120,266 across 198 hospitals
35 hospitals with n>=1000, covering 65,547 (54.5%) of the cohort
k=5 -> 50 joint cells


## 1. One codebook, fit on the pooled cohort, applied everywhere

In [2]:
def codebooks(ch, mod, seed):
    v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xv), K, seed=seed)
    s, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xs, sentinel=mod.TIME_SENTINEL), K, seed=seed)
    return v, s

# fit once per seed on the FULL eICU cohort; every subset below indexes into these
E_CODE = {sd: codebooks(cc_e, eicu, sd) for sd in SEEDS}
M_CODE = {sd: codebooks(cc_m, mimic, sd) for sd in SEEDS}
print(f"eICU codebooks fit on n={len(cc_e.y):,}; MIMIC on n={len(cc_m.y):,}")

def atoms(v, s, y):
    if len(np.unique(y)) < 2 or len(np.unique(v)) < 2 or len(np.unique(s)) < 2:
        return None
    d = pb.decompose(v, s, y, measure="broja")
    if not d["converged"]:
        return None
    a = d["atoms_pct_of_joint"]
    return {k: a[k] for k in ATOMS}

eICU codebooks fit on n=120,266; MIMIC on n=64,623


## 2. Within-hospital vs size-matched random slice

For each hospital and each codebook seed: the composition inside that hospital,
and the composition of `N_DRAWS` random draws of the same size taken from the
whole cohort. Averaged with hospital-size weights.

In [3]:
def target_mask_y(ch_groups, y_mort, target, demo):
    if target == "mortality":
        return np.ones(len(y_mort), bool), y_mort
    col = demo[target].reindex(pd.Index(ch_groups))
    m = col.notna().to_numpy()
    return m, col[m].to_numpy().astype(int)

rows = []
rng = np.random.default_rng(0)
for target in TARGETS:
    tmask, y_t = target_mask_y(cc_e.groups, cc_e.y, target, demo_e)
    h_t = hosp[tmask]
    for sd in SEEDS:
        v_all, s_all = E_CODE[sd]
        v_t, s_t = v_all[tmask], s_all[tmask]
        for h in BIG:
            inh = h_t == h
            n_h = int(inh.sum())
            if n_h < MIN_H // 2:
                continue
            a_in = atoms(v_t[inh], s_t[inh], y_t[inh])
            if a_in is None:
                continue
            ctrl = []
            for r in range(N_DRAWS):
                idx = rng.choice(len(y_t), n_h, replace=False)
                a_r = atoms(v_t[idx], s_t[idx], y_t[idx])
                if a_r is not None:
                    ctrl.append(a_r)
            if not ctrl:
                continue
            rows.append({"target": target, "seed": sd, "hospital": int(h), "n_h": n_h,
                         **{f"in_{k}": a_in[k] for k in ATOMS},
                         **{f"rand_{k}": float(np.mean([c[k] for c in ctrl])) for k in ATOMS}})
    print(f"  {target}: {sum(1 for r in rows if r['target']==target)} hospital x seed cells",
          flush=True)
wh = pd.DataFrame(rows)

  mortality: 105 hospital x seed cells


  gender_female: 105 hospital x seed cells


  age_ge65: 105 hospital x seed cells


  race_binary: 99 hospital x seed cells


## 3. The pooling effect: within-hospital minus size-matched random

In [4]:
def wavg(g, col):
    return np.average(g[col], weights=g.n_h)

out = []
for target, g in wh.groupby("target"):
    row = {"target": target, "n_cells": len(g), "median_n_h": int(g.n_h.median())}
    for k in ATOMS:
        row[f"within_{k}"] = wavg(g, f"in_{k}")
        row[f"random_{k}"] = wavg(g, f"rand_{k}")
        row[f"delta_{k}"] = row[f"within_{k}"] - row[f"random_{k}"]
    row["within_non_value"] = 100 - row["within_u_val"]
    row["random_non_value"] = 100 - row["random_u_val"]
    out.append(row)
pool = pd.DataFrame(out)

# u_str leads: it is the only atom that cannot be physiology, so it is the one
# the pooling question is actually about. u_val is kept for context, not as the
# headline -- see section 4.
print("size-matched, one shared codebook, k=5. positive delta = pooling was hiding it\n")
print(pool[["target","n_cells","median_n_h",
            "within_u_str","random_u_str","delta_u_str",
            "within_syn","random_syn","delta_syn",
            "within_u_val","random_u_val","delta_u_val"]]
      .sort_values("delta_u_str", ascending=False)
      .to_string(index=False, float_format=lambda x: f"{x:.2f}"))


size-matched, one shared codebook, k=5. positive delta = pooling was hiding it

       target  n_cells  median_n_h  within_u_str  random_u_str  delta_u_str  within_syn  random_syn  delta_syn  within_u_val  random_u_val  delta_u_val
  race_binary       99        1424          6.35          3.29         3.06       58.55       60.06      -1.51         21.85         25.55        -3.70
    mortality      105        1496          2.97          1.42         1.55       41.17       41.14       0.04         36.19         39.37        -3.18
     age_ge65      105        1496          0.82          0.08         0.74       33.81       27.46       6.35         51.23         62.06       -10.83
gender_female      105        1496          0.10          0.02         0.08       29.36       29.70      -0.34         63.87         64.62        -0.75


## 4. Does it close the gap to MIMIC-IV?

In [5]:
# Pooled reference: both databases decomposed on the whole cohort, same codebook
# family, so the within-hospital numbers have something to be compared against.
ref = []
for target in TARGETS:
    for db, cc, code_, demo in (("eICU-CRD pooled", cc_e, E_CODE, demo_e),
                                ("MIMIC-IV pooled", cc_m, M_CODE, demo_m)):
        tmask, y_t = target_mask_y(cc.groups, cc.y, target, demo)
        acc = []
        for sd in SEEDS:
            v_all, s_all = code_[sd]
            a = atoms(v_all[tmask], s_all[tmask], y_t)
            if a: acc.append(a)
        if acc:
            ref.append({"target": target, "db": db,
                        **{k: float(np.mean([x[k] for x in acc])) for k in ATOMS}})
ref = pd.DataFrame(ref)
print("pooled reference, k=5\n")
print(ref.pivot(index="target", columns="db", values="u_str")
        .to_string(float_format=lambda x: f"{x:.2f}"))
print()

# The gap analysis runs on u_str FIRST. The published version ran it only on
# u_val, which is the wrong axis for this question: u_val is physiology, and the
# claim under test is about the non-physiological component.
#
# The ratio is also gated. "gap closed %" divides by (eICU pooled - MIMIC
# pooled), and that denominator is near zero for most cells -- on u_val it is
# -4.78 for mortality, which turned a 17-point effect into "-357% closed". A
# ratio whose denominator is smaller than the noise describes the denominator,
# not the effect, so below 2 points it is reported as NaN and only the raw
# quantities are shown.
MIN_DENOM = 2.0

def gap_row(target, atom):
    e = ref[(ref.target == target) & (ref.db == "eICU-CRD pooled")]
    m = ref[(ref.target == target) & (ref.db == "MIMIC-IV pooled")]
    w = pool[pool.target == target]
    if e.empty or m.empty or w.empty:
        return None
    e_pool, m_pool = float(e[atom].iloc[0]), float(m[atom].iloc[0])
    within = float(w[f"within_{atom}"].iloc[0])
    denom = e_pool - m_pool
    closed = 100 * (e_pool - within) / denom if abs(denom) >= MIN_DENOM else float("nan")
    return {"target": target, "atom": atom, "eICU pooled": e_pool,
            "eICU within-hosp": within, "MIMIC pooled": m_pool,
            "within - pooled": within - e_pool, "gap closed %": closed}

comp = pd.DataFrame([r for target in TARGETS for atom in ("u_str", "u_val")
                     if (r := gap_row(target, atom)) is not None])
print("k=5, same codebook family. 'within - pooled' is the size-matched pooling")
print(f"effect and is the robust statistic; 'gap closed %' is NaN where the")
print(f"eICU/MIMIC pooled difference is under {MIN_DENOM} points.\n")
print(comp.to_string(index=False, float_format=lambda x: f"{x:.2f}", na_rep="--"))


pooled reference, k=5

db             MIMIC-IV pooled  eICU-CRD pooled
target                                         
age_ge65                 -0.00            -0.00
gender_female            -0.00            -0.00
mortality                 1.81             0.61
race_binary              30.62             0.95

k=5, same codebook family. 'within - pooled' is the size-matched pooling
effect and is the robust statistic; 'gap closed %' is NaN where the
eICU/MIMIC pooled difference is under 2.0 points.

       target  atom  eICU pooled  eICU within-hosp  MIMIC pooled  within - pooled  gap closed %
    mortality u_str         0.61              2.97          1.81             2.36            --
    mortality u_val        53.26             36.19         58.04           -17.07       -357.12
gender_female u_str        -0.00              0.10         -0.00             0.10            --
gender_female u_val        90.77             63.87         69.40           -26.90        125.90
     age_ge65 u_

## 5. Save

In [6]:
payload = {
    "question": "Is eICU-CRD's value-dominance a pooling artifact rather than a practice difference?",
    "design": {
        "k": K, "why_not_13": ("at k=13 a median eligible hospital has 4.4 patients per joint cell; "
                               "these numbers are NOT comparable to the k=13 headline"),
        "codebook": "fit once on the full pooled eICU cohort per seed, applied to every subset",
        "control": f"{N_DRAWS} size-matched random draws from the whole cohort per hospital",
        "min_hospital_n": MIN_H, "n_hospitals": len(BIG), "seeds": SEEDS,
        "coverage": float(vc[BIG].sum()/len(hosp))},
    "pooling_effect": pool.to_dict(orient="records"),
    "pooled_reference": ref.to_dict(orient="records"),
    "gap_closed": comp.to_dict(orient="records"),
}
with open("../results/eicu_within_hospital.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)
print("wrote results/eicu_within_hospital.json")

wrote results/eicu_within_hospital.json
